# Módulo AI7 · Inferencia y despliegue — Práctica (CPU)
Medimos lo que importa en producción: latencia, throughput con batching, KV-cache y cuantización int8 a mano.


In [ ]:
import time, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(42); np.random.seed(42)

# Modelo juguete (misma arquitectura del AI5): 2 capas, dim=64
class Bloque(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.qkv, self.proj = nn.Linear(dim, 3*dim), nn.Linear(dim, dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 4*dim), nn.GELU(), nn.Linear(4*dim, dim))
        self.ln1, self.ln2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.h = heads
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.h, -1).transpose(1, 2) for t in (q, k, v))
        a = (q @ k.transpose(-2, -1)) / math.sqrt(q.size(-1))
        a = a.masked_fill(torch.triu(torch.ones(T, T, dtype=torch.bool), 1), float("-inf"))
        a = (F.softmax(a, -1) @ v).transpose(1, 2).reshape(B, T, C)
        return x + self.mlp(self.ln2(x + self.proj(self.ln1(a))))

class Modelo(nn.Module):
    def __init__(self, vocab=100, dim=64):
        super().__init__()
        self.tok, self.pos = nn.Embedding(vocab, dim), nn.Embedding(64, dim)
        self.blocks = nn.Sequential(Bloque(dim, 4), Bloque(dim, 4))
        self.head = nn.Linear(dim, vocab)
    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1]))
        return self.head(self.blocks(x))

m = Modelo()


## 1 · Latencia y batching

In [ ]:
def inferencia(peticiones):
    """peticiones: lista de tensores (1, T) → tokens/s"""
    t0 = time.perf_counter()
    outs = [m(p).shape for p in peticiones]
    dt = time.perf_counter() - t0
    return len(peticiones) / dt, dt

def inferencia_batching(peticiones):
    T = min(p.shape[1] for p in peticiones)
    batch = torch.cat([p[:, :T] for p in peticiones])
    t0 = time.perf_counter()
    _ = m(batch)
    dt = time.perf_counter() - t0
    return len(peticiones) / dt, dt

peticiones = [torch.randint(0, 100, (1, 32)) for _ in range(8)]
for nombre, fn in [("sin batching", inferencia), ("con batching", inferencia_batching)]:
    pps, dt = fn(peticiones)
    print(f"{nombre:14s}: {dt*1000:7.1f} ms total, {pps*32:8.0f} tokens aprox/s")

**Lectura:** batching procesa 8 peticiones en ~1 pasada → throughput total ↑, pero la latencia de *cada* petición crece (espera al batch). En producción se equilibra con cola y TTL (vLLM lo hace por ti).

## 2 · KV-cache: por qué importa

In [ ]:
# Demo conceptual: coste de re-calcular el prefijo en cada paso (sin cache)
T = 32
costo_sin_cache = sum(t * (t + 1) / 2 for t in range(1, T + 1))     # O(T²)
costo_con_cache = T + T * (T - 1) // 2                              # solo nuevo token cuesta O(T) por paso… linear total
print(f"pares atendidos sin cache (T={T}): {costo_sin_cache:,}")
print(f"pares atendidos con cache (T={T}): {costo_con_cache:,}")
print(f"ratio ≈ {costo_sin_cache / costo_con_cache:.1f}× menos cómputo de atención")

## 3 · Cuantización int8 por canal, a mano


In [ ]:
W = torch.randn(256, 64)   # pesos de una proyección

def cuantizar_int8(W):
    """Cuantiza por canal (fila): escala = max|fila| / 127"""
    escala = W.abs().max(dim=1, keepdim=True).values / 127.0
    q = torch.round(W / escala).clamp(-127, 127).to(torch.int8)
    return q, escala

def dequantizar(q, escala):
    return q.float() * escala

q, escala = cuantizar_int8(W)
W_q = dequantizar(q, escala)
error_rel = (W - W_q).abs().mean() / W.abs().mean()
memoria_fp32, memoria_int8 = W.numel() * 4 / 1024, q.numel() / 1024
print(f"error relativo medio: {error_rel:.4%}")
print(f"memoria: fp32={memoria_fp32:.1f} KiB → int8={memoria_int8:.1f} KiB (4× menos)")

## 4 · Presupuesto de inferencia (sin API, concepto)

Con vLLM y un modelo 7B en cola, el coste por petición ≈ coste_hora / tokens_del_mes · tokens_por_petición.

El costo dominante: **KV-cache en memoria** (nº peticiones concurrentes × longitud). Cuantizar pesos baja VRAM del modelo, pero la KV-cache es fp16 → el cuello de la concurrencia. (PagedAttention en vLLM lo mitiga.)

---

## Ejercicios (4)
1. Batching: prueba con 32 peticiones de longitud 32 → tokens/s. ¿Sube linealmente? ¿Por qué no?
2. Cuantiza **global** (una escala para toda la W) vs por canal: compara error. ¿Cuál es mayor y por qué?
3. KV-cache: re-escribe el cálculo con T=64 — ¿el ratio baja o sube? (¿converge a qué?)
4. *(Colab)* Sirve el modelo `Qwen2.5-0.5B` con vLLM (`pip install vllm`, `vllm serve`) y mide tokens/s con 8 peticiones concurrentes (`curl`). Compara con tu juguete.
